<a href="https://colab.research.google.com/github/abhijadhav14/Data-Analytics-Using-Python/blob/main/sql2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder.appName("Student SQL Operations").getOrCreate()

In [3]:
data = [
    (1, "Arun", "CS", 85, 21),
    (2, "Bala", "IT", 78, 22),
    (3, "Charan", "CS", 92, 20),
    (4, "Divya", "ECE", 88, 21),
    (5, "Esha", "IT", 95, 23),
    (6, "Farhan", "ECE", 67, 22),
    (7, "Gita", "CS", 74, 21),
    (8, "Hari", "IT", 81, 20),
    (9, "Isha", "ECE", 90, 22),
    (10, "John", "CS", 60, 23)
]

columns = ["id", "name", "department", "marks", "age"]

In [4]:
df = spark.createDataFrame(data, columns)
df.createOrReplaceTempView("students")
print("Initial Data:")
df.show()

Initial Data:
+---+------+----------+-----+---+
| id|  name|department|marks|age|
+---+------+----------+-----+---+
|  1|  Arun|        CS|   85| 21|
|  2|  Bala|        IT|   78| 22|
|  3|Charan|        CS|   92| 20|
|  4| Divya|       ECE|   88| 21|
|  5|  Esha|        IT|   95| 23|
|  6|Farhan|       ECE|   67| 22|
|  7|  Gita|        CS|   74| 21|
|  8|  Hari|        IT|   81| 20|
|  9|  Isha|       ECE|   90| 22|
| 10|  John|        CS|   60| 23|
+---+------+----------+-----+---+



In [5]:
spark.sql("SELECT * FROM students").show()
spark.sql("SELECT name, marks FROM students").show()
spark.sql("SELECT * FROM students WHERE marks > 50").show()
spark.sql("SELECT * FROM students ORDER BY marks DESC").show()
spark.sql("SELECT * FROM students LIMIT 5").show()
spark.sql("SELECT * FROM students WHERE age > 22").show()

+---+------+----------+-----+---+
| id|  name|department|marks|age|
+---+------+----------+-----+---+
|  1|  Arun|        CS|   85| 21|
|  2|  Bala|        IT|   78| 22|
|  3|Charan|        CS|   92| 20|
|  4| Divya|       ECE|   88| 21|
|  5|  Esha|        IT|   95| 23|
|  6|Farhan|       ECE|   67| 22|
|  7|  Gita|        CS|   74| 21|
|  8|  Hari|        IT|   81| 20|
|  9|  Isha|       ECE|   90| 22|
| 10|  John|        CS|   60| 23|
+---+------+----------+-----+---+

+------+-----+
|  name|marks|
+------+-----+
|  Arun|   85|
|  Bala|   78|
|Charan|   92|
| Divya|   88|
|  Esha|   95|
|Farhan|   67|
|  Gita|   74|
|  Hari|   81|
|  Isha|   90|
|  John|   60|
+------+-----+

+---+------+----------+-----+---+
| id|  name|department|marks|age|
+---+------+----------+-----+---+
|  1|  Arun|        CS|   85| 21|
|  2|  Bala|        IT|   78| 22|
|  3|Charan|        CS|   92| 20|
|  4| Divya|       ECE|   88| 21|
|  5|  Esha|        IT|   95| 23|
|  6|Farhan|       ECE|   67| 22|
|  7| 

In [6]:
spark.sql("SELECT department, AVG(marks) AS avg_marks FROM students GROUP BY department").show()
spark.sql("SELECT department, MAX(marks) AS max_marks FROM students GROUP BY department").show()
spark.sql("SELECT department, COUNT(*) AS total_students FROM students GROUP BY department").show()

+----------+-----------------+
|department|        avg_marks|
+----------+-----------------+
|       ECE|81.66666666666667|
|        IT|84.66666666666667|
|        CS|            77.75|
+----------+-----------------+

+----------+---------+
|department|max_marks|
+----------+---------+
|       ECE|       90|
|        IT|       95|
|        CS|       92|
+----------+---------+

+----------+--------------+
|department|total_students|
+----------+--------------+
|       ECE|             3|
|        IT|             3|
|        CS|             4|
+----------+--------------+



In [7]:
spark.sql("""
SELECT department, AVG(marks) AS avg_marks
FROM students
GROUP BY department
HAVING avg_marks > 80
""").show()

+----------+-----------------+
|department|        avg_marks|
+----------+-----------------+
|       ECE|81.66666666666667|
|        IT|84.66666666666667|
+----------+-----------------+



In [8]:
spark.sql("""
SELECT name, marks,
CASE
    WHEN marks >= 90 THEN 'Excellent'
    WHEN marks >= 75 THEN 'Good'
    ELSE 'Average'
END AS grade
FROM students
""").show()

+------+-----+---------+
|  name|marks|    grade|
+------+-----+---------+
|  Arun|   85|     Good|
|  Bala|   78|     Good|
|Charan|   92|Excellent|
| Divya|   88|     Good|
|  Esha|   95|Excellent|
|Farhan|   67|  Average|
|  Gita|   74|  Average|
|  Hari|   81|     Good|
|  Isha|   90|Excellent|
|  John|   60|  Average|
+------+-----+---------+



In [9]:
spark.sql("SELECT DISTINCT department FROM students").show()

+----------+
|department|
+----------+
|       ECE|
|        IT|
|        CS|
+----------+



In [10]:
spark.sql("""
SELECT * FROM students
WHERE marks > (SELECT AVG(marks) FROM students)
""").show()

+---+------+----------+-----+---+
| id|  name|department|marks|age|
+---+------+----------+-----+---+
|  1|  Arun|        CS|   85| 21|
|  3|Charan|        CS|   92| 20|
|  4| Divya|       ECE|   88| 21|
|  5|  Esha|        IT|   95| 23|
|  9|  Isha|       ECE|   90| 22|
+---+------+----------+-----+---+



In [11]:
spark.sql("""
SELECT name, department, marks,
RANK() OVER (PARTITION BY department ORDER BY marks DESC) AS rank
FROM students
""").show()

+------+----------+-----+----+
|  name|department|marks|rank|
+------+----------+-----+----+
|Charan|        CS|   92|   1|
|  Arun|        CS|   85|   2|
|  Gita|        CS|   74|   3|
|  John|        CS|   60|   4|
|  Isha|       ECE|   90|   1|
| Divya|       ECE|   88|   2|
|Farhan|       ECE|   67|   3|
|  Esha|        IT|   95|   1|
|  Hari|        IT|   81|   2|
|  Bala|        IT|   78|   3|
+------+----------+-----+----+



In [12]:
spark.sql("""
SELECT a.name AS student1, b.name AS student2, a.department
FROM students a
JOIN students b
ON a.department = b.department AND a.id != b.id
""").show()

+--------+--------+----------+
|student1|student2|department|
+--------+--------+----------+
|    Arun|  Charan|        CS|
|    Arun|    Gita|        CS|
|    Arun|    John|        CS|
|  Charan|    Arun|        CS|
|  Charan|    Gita|        CS|
|  Charan|    John|        CS|
|    Gita|    Arun|        CS|
|    Gita|  Charan|        CS|
|    Gita|    John|        CS|
|    John|    Arun|        CS|
|    John|  Charan|        CS|
|    John|    Gita|        CS|
|   Divya|  Farhan|       ECE|
|   Divya|    Isha|       ECE|
|  Farhan|   Divya|       ECE|
|  Farhan|    Isha|       ECE|
|    Isha|   Divya|       ECE|
|    Isha|  Farhan|       ECE|
|    Bala|    Esha|        IT|
|    Bala|    Hari|        IT|
+--------+--------+----------+
only showing top 20 rows


In [13]:
dept_data = [
    ("CS", "Computer Science"),
    ("IT", "Information Technology"),
    ("ECE", "Electronics")
]

dept_df = spark.createDataFrame(dept_data, ["dept_code", "dept_name"])
dept_df.createOrReplaceTempView("departments")

In [14]:
spark.sql("""
SELECT s.name, s.marks, d.dept_name
FROM students s
JOIN departments d
ON s.department = d.dept_code
""").show()

+------+-----+--------------------+
|  name|marks|           dept_name|
+------+-----+--------------------+
|  Arun|   85|    Computer Science|
|Charan|   92|    Computer Science|
|  Gita|   74|    Computer Science|
|  John|   60|    Computer Science|
| Divya|   88|         Electronics|
|Farhan|   67|         Electronics|
|  Isha|   90|         Electronics|
|  Bala|   78|Information Techn...|
|  Esha|   95|Information Techn...|
|  Hari|   81|Information Techn...|
+------+-----+--------------------+



In [15]:
spark.sql("""
SELECT s.name, d.dept_name
FROM students s
LEFT JOIN departments d
ON s.department = d.dept_code
""").show()

+------+--------------------+
|  name|           dept_name|
+------+--------------------+
| Divya|         Electronics|
|  Bala|Information Techn...|
|  Esha|Information Techn...|
|  Arun|    Computer Science|
|Charan|    Computer Science|
|Farhan|         Electronics|
|  Isha|         Electronics|
|  Hari|Information Techn...|
|  Gita|    Computer Science|
|  John|    Computer Science|
+------+--------------------+



In [16]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW high_scorers AS
SELECT * FROM students WHERE marks > 85
""")

DataFrame[]

In [17]:
spark.sql("SELECT * FROM high_scorers").show()

+---+------+----------+-----+---+
| id|  name|department|marks|age|
+---+------+----------+-----+---+
|  3|Charan|        CS|   92| 20|
|  4| Divya|       ECE|   88| 21|
|  5|  Esha|        IT|   95| 23|
|  9|  Isha|       ECE|   90| 22|
+---+------+----------+-----+---+



In [18]:
spark.sql("""
SELECT name FROM students WHERE department='CS'
UNION
SELECT name FROM students WHERE department='IT'
""").show()

+------+
|  name|
+------+
|  Arun|
|Charan|
|  Gita|
|  John|
|  Esha|
|  Bala|
|  Hari|
+------+



In [19]:
spark.catalog.dropTempView("high_scorers")

spark.stop()